In [ ]:
import sys
if 'google.colab' in sys.modules:
    from IPython.core.getipython import get_ipython
    get_ipython().run_line_magic("pip", "install transformers sentencepiece accelerate")

In [ ]:
from transformers import pipeline
classifier = pipeline('zero-shot-classification', model='roberta-large-mnli')

In [ ]:
import torch
import activation_additions as aa
import random as pyrandom

import os
import csv

from typing import Dict, Union, Callable, Tuple, List
from functools import partial
from transformers import AutoModelForCausalLM, AutoTokenizer
from activation_additions.compat import get_x_vector, get_n_steered_completions, get_n_baseline_completions
import matplotlib.pyplot as plt
from numpy import random

import nltk
from tqdm.notebook import tqdm

In [ ]:
device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"
_ = torch.set_grad_enabled(False)
device

In [ ]:
nltk.download('punkt_tab')

In [ ]:
MODEL = "openai-community/gpt2-xl"
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model.to_str_tokens = lambda t: [t.replace('Ġ', ' ') for t in tokenizer.tokenize(t)]
model.tokenizer = tokenizer
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
NUM_BLOCKS = len(aa.get_blocks(model))

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "tokens_to_generate": 50,
    "seed": 0,
}
get_x_vector_preset: Callable = partial(
    get_x_vector,
    pad_method="tokens_right",
    model=model,
    custom_pad_id=tokenizer.pad_token_id,
)

In [ ]:
from pandas import read_csv
csv_file = read_csv("data_hub\concepts.csv").to_dict()
concept_indexed = {}

for j in csv_file["Concept"]:
    cur_data = {}
    for i in csv_file:
        cur_data[i] = csv_file[i][j]
    concept_indexed[csv_file["Concept"][j]] = cur_data
print(concept_indexed)

In [ ]:
def get_prompt_add(concept: str, s_type=1) -> str:
    return concept_indexed[concept]["Positive Prompt Type "+str(s_type)]
    
def get_prompt_sub(concept: str,s_type=1) -> str:
    return concept_indexed[concept]["Negative Prompt Type "+str(s_type)]

## Experiments with a Bert Based Model

In [ ]:
def get_pos_label(concept):
    return concept_indexed[concept]["Positive Label"] + "."
def get_neg_label(concept):
    return concept_indexed[concept]["Negative Label"] + "." 

def evaluate_matching_with_bert(concept, text):
    candidate_labels = [get_pos_label(concept),get_neg_label(concept)]
    classified = classifier(text, candidate_labels)
    if classified["labels"][0] == get_pos_label(concept):
        return classified["scores"][0]
    else: return classified["scores"][1]

In [ ]:
def write_to_csv(file_path: str, row_to_write: List[str]):
    with open(file_path, "a", newline="", encoding="utf-8") as f:
        result_file_writer = csv.writer(f)
        result_file_writer.writerow(row_to_write)
        # Usefull when running in collab:
        # f.flush()
        # os.fsync(f.fileno())

In [ ]:
def get_average_score_for_completions(completions, concept):
    sum_score = 0
    for completion in completions:
        current_score = evaluate_matching_with_bert(concept, completion)
        sum_score += current_score
    
    avg_score = sum_score / len(completions)

    return avg_score

def get_average_score(
    prompts: List[str],
    num_completions: int,
    concept: str,
    coeff: float,
    layer: int,
    prompt_type: int, 
    **sampling_kwargs
):
    num_prompts = len(prompts)

    assert num_completions % num_prompts == 0
   
    pos_prompt, neg_prompt = get_prompt_add(concept, prompt_type), get_prompt_sub(concept, prompt_type)

    additions = get_x_vector_preset(prompt1=pos_prompt, prompt2=neg_prompt, coeff=coeff, act_name=layer)

    

    # get the completions with the steered model
    
    completions = []
    for i in range(num_prompts):
        # get completions for each prompt
        completions += get_n_steered_completions([prompts[i]] * (num_completions // num_prompts), model, additions, **sampling_kwargs)



    avg_score = get_average_score_for_completions(completions, concept)

    return avg_score, pyrandom.choice(completions) # return a random completion for sanity checks


def find_optimal_layer(
    prompts: List[str],
    num_completions: int,
    concept: str,
    coeff: float,
    prompt_type: int,
    file_path: str,
    **sampling_kwargs
):
    scores = []
    for layer in tqdm(range(NUM_BLOCKS), total=NUM_BLOCKS):
        s, _ = get_average_score(prompts, num_completions, concept, coeff, layer, prompt_type, **sampling_kwargs)
        scores.append(s)

    scores_tensor = torch.tensor(scores)
    optimal_layer = int(torch.argmax(scores_tensor).item())

    optimal_score = scores[optimal_layer]
    
    # save the result in a file
    write_to_csv(file_path, [concept, f"{prompt_type}" , f"{coeff}", f"{optimal_layer}", f"{optimal_score}"] )

    plt.figure()
    plt.plot(range(NUM_BLOCKS), scores)
    plt.xlabel("Layer")
    plt.ylabel("Score")
    plt.title(f"{concept} with type {prompt_type}")
    os.makedirs("optimal_layer_graphs", exist_ok=True)
    plt.savefig(f"optimal_layer_graphs/{concept}_{prompt_type}.png")
    plt.close()

    return optimal_layer

In [ ]:

def get_baseline_completions(
    prompts: List[str],
    num_completions: int,    
    **sampling_kwargs    
):
    num_prompts = len(prompts)
    assert num_completions % num_prompts == 0
    
    completions = []
    for i in range(num_prompts):
        completions += get_n_baseline_completions([prompts[i]] * (num_completions // num_prompts), model, **sampling_kwargs)
    
    return completions

In [ ]:
len(concept_indexed.keys())

#### Find optimal layers

In [ ]:
prompts = ["I was just thinking about something. Did you know that",
            "I was talking to somebody the other day, and they were telling me that" ]
coeff=2


# number of completions per experiment
num_completions = 20

write_to_csv("optimal_parameters.csv", ["Concept", "Prompt Type" , "Coefficient", "Optimal Layer", "Optimal Score"])

concept_range = (0,1)
for ind,concept in enumerate(list(concept_indexed.keys())[concept_range[0]:concept_range[1]]):
    print("Running concept "+str(ind+concept_range[0])+"/"+str(concept_range[1]-1))
    for prompt_type in [1,2]:
        find_optimal_layer(prompts=prompts,
                            num_completions=num_completions,
                            concept=concept,
                            coeff=coeff,
                            prompt_type=prompt_type,
                            file_path="optimal_parameters.csv",
                            **sampling_kwargs)
    

In [ ]:
def read_optimal_layers(file_path: str) -> Dict[Tuple[str, int], int]:
    """
    CSV columns: concept,optimal_layer,prompt_type  (prompt_type is 1 or 2)
    Returns: {(concept, prompt_type): optimal_layer}
    Example: {("wedding", 1): 12, ("wedding", 2): 18}
    """
    out: Dict[Tuple[str, int], int] = {}

    with open(file_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        
        for row in reader:
            concept = (row.get("Concept") or "").strip()
            if not concept:
                continue
            
            ptype = int(str(row.get("Prompt Type", "")).strip())
            
            
            layer = int(str(row.get("Optimal Layer", "")).strip())
            
            key = (concept, ptype)
            
            out[key] = layer

    return out

#### Calculate Scores with the BERT based Model

In [ ]:
with open("data_hub\evaluation_prompts.txt") as file:
    prompts = [line.rstrip() for line in file]

print("Number of prompts: ",len(prompts))
print(prompts)

In [ ]:
coeff=2

# number of completions per experiment
num_completions = 100

baseline_completions = get_baseline_completions(prompts, num_completions, **sampling_kwargs)

# maps (concept, prompt_type) to optimal layer
optimal_layers = read_optimal_layers("optimal_parameters.csv")




# Write final results to file
write_to_csv("bert_scores.csv", 
             ["Concept", "Prompt Type" , "Coefficient", "Optimal Layer", "Final Score", "Steered Score", "Baseline Score", "Random Completion"])

concept_range = (0,0)
for ind,concept in enumerate(list(concept_indexed.keys())[concept_range[0]:concept_range[1]]):
    print("Running concept "+str(ind+concept_range[0])+"/"+str(concept_range[1]-1))

    for prompt_type in [1,2]:
        optimal_layer = optimal_layers[(concept, prompt_type)]
        
        
        steered_score, random_completion = get_average_score(prompts, num_completions, concept, coeff, optimal_layer, prompt_type, **sampling_kwargs)
        baseline_score = get_average_score_for_completions(baseline_completions, concept)

        final_score = steered_score - baseline_score

        write_to_csv("bert_scores.csv", [concept, f"{prompt_type}" , f"{coeff}", f"{optimal_layer}", f"{final_score}", f"{steered_score}", f"{baseline_score}", f"{random_completion}"])
